# Notebook 6f — Full-Mix Diagrams (every chart from NB6e, generated separately)

This notebook does **no training**. It reloads the trained `fullmix_model.pt`
checkpoint from NB6e, rebuilds the exact same test splits (same `SEED=42`
record-level split logic, so the data is identical), runs inference once, and
then renders **every diagram from NB6e as its own standalone figure** — one
plot per cell, one PNG per plot — instead of the combined multi-panel figures
NB6e produces.

**Prerequisites**
- NB6e already run successfully -> `./ECG_Project/data/fullmix_model.pt` exists
- The same raw dataset folders (INCART, CPSC 2018, Chapman-Shaoxing) used in NB6e are still reachable (override `*_ROOT` below if auto-detect fails)
- *(Optional, for the 3 training-curve diagrams)* — see the note in **Step 6** below: NB6e doesn't persist its epoch-by-epoch history by default, so add the one small cell shown there to NB6e and rerun it once if you want those 3 curves here too. Everything else works without it.

**Outputs** — 23 individually-saved PNGs, each prefixed `fullmix_diag_*.png`, in `./ECG_Project/data/diagrams/`.


In [ ]:
# Environment safety (Windows OpenMP / pin_memory fixes)
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys, json, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import wfdb
from scipy.signal import resample as scipy_resample

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, average_precision_score,
    precision_score, recall_score, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = (device.type == "cuda")

SAVE_DIR = Path("./ECG_Project/data")
DIAG_DIR = SAVE_DIR / "diagrams"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SAVE_DIR))

from ecg_model import ECGRiskNetXAI

meta_path = SAVE_DIR / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError("metadata.json not found. Run NB1 first.")
with open(meta_path) as f:
    META = json.load(f)

RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
SNOMED_TO_RISK = META.get("snomed_to_risk", {})
STANDARD_LEAD_ORDER = META.get("standard_lead_order",
    ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"])
TARGET_LEN = META.get("target_len", 1000)
CLASS_NAMES = ["Low", "Moderate", "High", "Critical"]
CLASS_COLORS = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]

fullmix_ckpt = SAVE_DIR / "fullmix_model.pt"
if not fullmix_ckpt.exists():
    raise FileNotFoundError(
        "fullmix_model.pt not found. Run NB6e (Full-Mix Training) first."
    )

print("Setup complete.")
print(f"Device      : {device}")
print(f"Diagrams -> : {DIAG_DIR.resolve()}")

## Step 1 — Shared loader utilities (identical to NB6e, for reproducible splits)

In [ ]:
def reorder_leads(signal, sig_names, target_order=STANDARD_LEAD_ORDER):
    name_to_idx = {}
    for i, name in enumerate(sig_names):
        if isinstance(name, (bytes, bytearray)):
            try:
                n = name.decode("utf-8").upper()
            except Exception:
                n = str(name).upper()
        else:
            n = str(name).upper()
        name_to_idx[n] = i
    missing = [lead for lead in target_order if lead.upper() not in name_to_idx]
    if missing:
        raise ValueError(f"Missing leads {missing} in record leads: {sig_names}")
    idx_order = [name_to_idx[lead.upper()] for lead in target_order]
    return signal[:, idx_order]


def split_records_3way(records, seed=SEED, train_frac=0.70, val_frac=0.15):
    """Identical logic/seed to NB6e -- reproduces the exact same test set."""
    records = list(records)
    rng = np.random.RandomState(seed)
    shuffled = records.copy()
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(train_frac * n))
    n_val = max(1, int(val_frac * n))
    train_set = set(shuffled[:n_train])
    val_set = set(shuffled[n_train:n_train + n_val])
    test_set = set(shuffled[n_train + n_val:])
    if len(test_set) == 0 and len(val_set) > 1:
        test_set.add(val_set.pop())
    return train_set, val_set, test_set


def stack_segments(segments):
    if len(segments) == 0:
        return None, None, None
    X = np.stack([s[0] for s in segments]).astype(np.float32)
    y = np.array([s[1] for s in segments], dtype=np.int64)
    M = np.stack([s[2] for s in segments]).astype(np.float32)
    return X, y, M


print("Shared utilities ready (same seed/logic as NB6e -> identical test splits).")

## Step 2 — PTB-XL test set (from NB1/NB2 processed files)

In [ ]:
ptb_test_path = SAVE_DIR / "test_processed.npz"
if not ptb_test_path.exists():
    raise FileNotFoundError("test_processed.npz not found. Run NB1 + NB2 first.")

d_te = np.load(ptb_test_path)
X_ptb_test = d_te["X"].astype(np.float32)
y_ptb_test = d_te["y"].astype(np.int64)
M_ptb_test = d_te["meta"].astype(np.float32)

print(f"PTB-XL test : {X_ptb_test.shape}")

## Step 3 — INCART test set (AAMI annotations, same record-level split as NB6e)

In [ ]:
INCART_ROOT = None   # override if auto-detect fails, e.g. r"D:/datasets/incart/Training"

candidates = []
cwd = Path.cwd()
ancestors = [cwd] + list(cwd.parents[:6])
for root in ancestors:
    candidates.extend([
        root / "data" / "incart" / "Training",
        root / "ECG_Project" / "data" / "incart" / "Training",
        root / "ECG_Project" / "data" / "incart",
        root / "data" / "incart",
    ])
try:
    candidates.extend([SAVE_DIR / "incart" / "Training", SAVE_DIR / "incart", SAVE_DIR.parent / "incart" / "Training"])
except Exception:
    pass

seen, uniq_candidates = set(), []
for p in candidates:
    p = Path(p)
    if str(p) not in seen:
        seen.add(str(p)); uniq_candidates.append(p)

incart_path = None
for p in uniq_candidates:
    if p.exists() and any(p.glob("*.hea")):
        incart_path = p
        break
if incart_path is None:
    hea_files = list(Path(".").rglob("*.hea")) + list(Path(".").rglob("*.hea.gz"))
    incart_hits = [f for f in hea_files if "incart" in str(f).lower()]
    search_set = incart_hits if incart_hits else hea_files
    if search_set:
        parents = [f.parent for f in search_set]
        incart_path = max(set(parents), key=lambda d: parents.count(d))
if INCART_ROOT:
    incart_path = Path(INCART_ROOT)
if incart_path is None or not incart_path.exists():
    raise FileNotFoundError("INCART folder not found. Set INCART_ROOT above.")

hea_list = sorted(list(incart_path.glob("*.hea")) + list(incart_path.glob("*.hea.gz")))
ALL_INCART_RECORDS = [p.stem for p in hea_list]
print(f"INCART path: {incart_path.resolve()}  |  records: {len(ALL_INCART_RECORDS)}")

SYMBOL_TO_AAMI = {
    "N": "N", ".": "N", "e": "N", "j": "N", "L": "N", "R": "N",
    "A": "S", "a": "S", "J": "S", "S": "S", "V": "V", "E": "V", "F": "F", "P": "N",
    "/": "Q", "Q": "Q", "?": "Q", "[": "Q", "]": "Q", "!": "Q", "f": "Q",
}
AAMI_TO_RISK = {"N": 0, "S": 1, "V": 2, "F": 2, "Q": 3}


def _normalize_symbol(sym):
    if sym is None:
        return ""
    if isinstance(sym, (bytes, bytearray)):
        try:
            s = sym.decode("utf-8")
        except Exception:
            s = str(sym)
    else:
        s = str(sym)
    s = s.strip()
    if s == "" or s in SYMBOL_TO_AAMI:
        return s
    if s.upper() in SYMBOL_TO_AAMI:
        return s.upper()
    if s[0] in SYMBOL_TO_AAMI:
        return s[0]
    if s[0].upper() in SYMBOL_TO_AAMI:
        return s[0].upper()
    return s


def get_incart_segments(record_path):
    try:
        rec = wfdb.rdrecord(str(record_path))
        ann = wfdb.rdann(str(record_path), "atr")
    except Exception:
        return []
    fs = getattr(rec, "fs", None)
    if fs is None:
        return []
    try:
        signal = reorder_leads(rec.p_signal, rec.sig_name)
    except ValueError:
        return []
    n_samp = signal.shape[0]
    seg_len_orig = int(10 * fs)
    results = []
    ann_samples = list(getattr(ann, "sample", []))
    ann_symbols = list(getattr(ann, "symbol", []))
    for start in range(0, max(1, n_samp - seg_len_orig), seg_len_orig):
        end = start + seg_len_orig
        if end > n_samp:
            break
        seg = signal[start:end, :]
        ann_in_window = []
        for i, sidx in enumerate(ann_samples):
            if start <= sidx < end:
                sym_raw = ann_symbols[i] if i < len(ann_symbols) else ""
                ann_in_window.append(_normalize_symbol(sym_raw))
        if not ann_in_window:
            continue
        max_risk = 0
        for sym in ann_in_window:
            aami = SYMBOL_TO_AAMI.get(sym, SYMBOL_TO_AAMI.get(sym.upper(), "N"))
            risk = AAMI_TO_RISK.get(aami, 0)
            if risk > max_risk:
                max_risk = risk
        try:
            seg_resampled = scipy_resample(seg, TARGET_LEN, axis=0)
        except Exception:
            continue
        mean = np.mean(seg_resampled, axis=0, keepdims=True)
        std = np.std(seg_resampled, axis=0, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        seg_norm = np.nan_to_num((seg_resampled - mean) / std, nan=0.0, posinf=0.0, neginf=0.0)
        meta = np.array([0.6, 0.5, 0.375], dtype=np.float32)
        results.append((seg_norm.astype(np.float32), max_risk, meta))
    return results


_, _, incart_test_recs = split_records_3way(ALL_INCART_RECORDS)
incart_test_seg = []
print("Loading INCART test records...")
for rec_name in ALL_INCART_RECORDS:
    if rec_name in incart_test_recs:
        incart_test_seg.extend(get_incart_segments(incart_path / rec_name))

X_incart_test, y_incart_test, M_incart_test = stack_segments(incart_test_seg)
if X_incart_test is None:
    raise RuntimeError("INCART test split produced no segments.")
print(f"INCART test : {X_incart_test.shape}")
del incart_test_seg
gc.collect()

## Step 4 — CPSC 2018 + Chapman-Shaoxing shared SNOMED loader

In [ ]:
def _parse_header_comments(rec):
    age, sex, dx_codes = None, None, []
    for line in getattr(rec, "comments", []) or []:
        line = line.strip()
        low = line.lower()
        if low.startswith("age"):
            try:
                val = line.split(":", 1)[1].strip()
                age = float(val) if val.replace(".", "", 1).isdigit() else None
            except Exception:
                age = None
        elif low.startswith("sex"):
            try:
                val = line.split(":", 1)[1].strip().lower()
                if val.startswith("m"):
                    sex = 1.0
                elif val.startswith("f"):
                    sex = 0.0
            except Exception:
                sex = None
        elif low.startswith("dx"):
            try:
                val = line.split(":", 1)[1].strip()
                dx_codes = [c.strip() for c in val.split(",") if c.strip()]
            except Exception:
                dx_codes = []
    return age, sex, dx_codes


_EXTRA_SNOMED = {
    "713427006": 1, "164909002": 1, "445118002": 1, "251120003": 1,
    "6374002": 1, "270492004": 1, "195042002": 1, "428750005": 1,
    "164934002": 1, "164931005": 1, "251200008": 1, "11092001": 1,
}
_EFFECTIVE_SNOMED = {**_EXTRA_SNOMED, **SNOMED_TO_RISK}


def get_snomed_wfdb_segments(record_path):
    try:
        rec = wfdb.rdrecord(str(record_path))
    except Exception:
        return []
    fs = getattr(rec, "fs", None)
    if fs is None:
        return []
    try:
        signal = reorder_leads(rec.p_signal, rec.sig_name)
    except ValueError:
        return []
    age, sex, dx_codes = _parse_header_comments(rec)
    if not dx_codes:
        return []
    max_risk, any_mapped = -1, False
    for code in dx_codes:
        if code in _EFFECTIVE_SNOMED:
            any_mapped = True
            risk = _EFFECTIVE_SNOMED[code]
            if risk > max_risk:
                max_risk = risk
    if not any_mapped:
        return []
    n_samp = signal.shape[0]
    seg_len_orig = int(10 * fs)
    if n_samp < seg_len_orig:
        return []
    age_norm = (age / 100.0) if age is not None else 0.6
    sex_norm = sex if sex is not None else 0.5
    meta = np.array([age_norm, sex_norm, 0.375], dtype=np.float32)
    results = []
    for start in range(0, max(1, n_samp - seg_len_orig + 1), seg_len_orig):
        end = start + seg_len_orig
        if end > n_samp:
            break
        seg = signal[start:end, :]
        try:
            seg_resampled = scipy_resample(seg, TARGET_LEN, axis=0)
        except Exception:
            continue
        mean = np.mean(seg_resampled, axis=0, keepdims=True)
        std = np.std(seg_resampled, axis=0, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        seg_norm = np.nan_to_num((seg_resampled - mean) / std, nan=0.0, posinf=0.0, neginf=0.0)
        results.append((seg_norm.astype(np.float32), max_risk, meta))
    return results


def discover_records(root_path):
    return sorted(list(Path(root_path).rglob("*.hea")))


print("Shared SNOMED/WFDB loader ready.")

## Step 5 — CPSC 2018 and Chapman-Shaoxing test sets

In [ ]:
CPSC_ROOT = None   # override if auto-detect fails

cpsc_candidates = []
for root in ancestors:
    cpsc_candidates.extend([
        root / "data" / "cpsc_2018", root / "data" / "cpsc2018",
        root / "ECG_Project" / "data" / "cpsc_2018", root / "ECG_Project" / "data" / "cpsc2018",
    ])
try:
    cpsc_candidates.extend([SAVE_DIR / "cpsc_2018", SAVE_DIR / "cpsc2018", SAVE_DIR.parent / "cpsc_2018"])
except Exception:
    pass

seen, uniq_cpsc = set(), []
for p in cpsc_candidates:
    p = Path(p)
    if str(p) not in seen:
        seen.add(str(p)); uniq_cpsc.append(p)

cpsc_path = None
for p in uniq_cpsc:
    if p.exists() and len(list(p.rglob("*.hea"))) > 0:
        cpsc_path = p
        break
if cpsc_path is None:
    hea_files = list(Path(".").rglob("*.hea"))
    cpsc_hits = [f for f in hea_files if "cpsc" in str(f).lower()]
    if cpsc_hits:
        parents = [f.parent.parent if f.parent.name.startswith("g") else f.parent for f in cpsc_hits]
        cpsc_path = max(set(parents), key=lambda d: parents.count(d))
if CPSC_ROOT:
    cpsc_path = Path(CPSC_ROOT)
if cpsc_path is None or not cpsc_path.exists():
    raise FileNotFoundError("CPSC 2018 folder not found. Set CPSC_ROOT above.")

cpsc_hea_files = discover_records(cpsc_path)
print(f"CPSC 2018 path: {cpsc_path.resolve()}  |  records: {len(cpsc_hea_files)}")

_, _, cpsc_test_recs = split_records_3way(cpsc_hea_files)
cpsc_test_seg = []
print("Loading CPSC 2018 test records...")
for hea_path in cpsc_hea_files:
    if hea_path in cpsc_test_recs:
        cpsc_test_seg.extend(get_snomed_wfdb_segments(hea_path.with_suffix("")))

X_cpsc_test, y_cpsc_test, M_cpsc_test = stack_segments(cpsc_test_seg)
if X_cpsc_test is None:
    raise RuntimeError("CPSC 2018 test split produced no segments.")
print(f"CPSC 2018 test : {X_cpsc_test.shape}")
del cpsc_test_seg
gc.collect()

# -- Chapman-Shaoxing --
CHAPMAN_ROOT = None   # override if auto-detect fails

chapman_candidates = []
for root in ancestors:
    chapman_candidates.extend([
        root / "data" / "chapman_shaoxing", root / "data" / "chapman",
        root / "ECG_Project" / "data" / "chapman_shaoxing", root / "ECG_Project" / "data" / "chapman",
    ])
try:
    chapman_candidates.extend([SAVE_DIR / "chapman_shaoxing", SAVE_DIR / "chapman", SAVE_DIR.parent / "chapman_shaoxing"])
except Exception:
    pass

seen, uniq_chapman = set(), []
for p in chapman_candidates:
    p = Path(p)
    if str(p) not in seen:
        seen.add(str(p)); uniq_chapman.append(p)

chapman_path = None
for p in uniq_chapman:
    if p.exists() and len(list(p.rglob("*.hea"))) > 0:
        chapman_path = p
        break
if chapman_path is None:
    hea_files = list(Path(".").rglob("*.hea"))
    chapman_hits = [f for f in hea_files if "chapman" in str(f).lower()]
    if chapman_hits:
        parents = [f.parent.parent if f.parent.name.startswith("g") else f.parent for f in chapman_hits]
        chapman_path = max(set(parents), key=lambda d: parents.count(d))
if CHAPMAN_ROOT:
    chapman_path = Path(CHAPMAN_ROOT)
if chapman_path is None or not chapman_path.exists():
    raise FileNotFoundError("Chapman-Shaoxing folder not found. Set CHAPMAN_ROOT above.")

chapman_hea_files = discover_records(chapman_path)
print(f"Chapman path: {chapman_path.resolve()}  |  records: {len(chapman_hea_files)}")

# Keep CHAPMAN_MAX_RECORDS in sync with NB6e if you subsampled there, so the
# split is reproduced identically.
CHAPMAN_MAX_RECORDS = None
chapman_records_to_use = chapman_hea_files
if CHAPMAN_MAX_RECORDS is not None:
    rng_ch = np.random.RandomState(SEED)
    idx = rng_ch.permutation(len(chapman_hea_files))[:CHAPMAN_MAX_RECORDS]
    chapman_records_to_use = [chapman_hea_files[i] for i in sorted(idx)]

_, _, chapman_test_recs = split_records_3way(chapman_records_to_use)
chapman_test_seg = []
print("Loading Chapman-Shaoxing test records...")
for hea_path in chapman_records_to_use:
    if hea_path in chapman_test_recs:
        chapman_test_seg.extend(get_snomed_wfdb_segments(hea_path.with_suffix("")))

X_chapman_test, y_chapman_test, M_chapman_test = stack_segments(chapman_test_seg)
if X_chapman_test is None:
    raise RuntimeError("Chapman-Shaoxing test split produced no segments.")
print(f"Chapman test : {X_chapman_test.shape}")
del chapman_test_seg
gc.collect()

## Step 6 — Pool the four test sets + load the trained Full-Mix model

In [ ]:
DATASET_NAMES = ["PTB-XL", "INCART", "CPSC2018", "Chapman"]

X_mix_test = np.concatenate([X_ptb_test, X_incart_test, X_cpsc_test, X_chapman_test], axis=0)
y_mix_test = np.concatenate([y_ptb_test, y_incart_test, y_cpsc_test, y_chapman_test], axis=0)
M_mix_test = np.concatenate([M_ptb_test, M_incart_test, M_cpsc_test, M_chapman_test], axis=0)

print(f"Pooled TEST : X={X_mix_test.shape}  y={y_mix_test.shape}")

model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3).to(device)
model.load_state_dict(torch.load(fullmix_ckpt, map_location=device))
model.eval()
print("fullmix_model.pt loaded.")

# Optional: epoch-by-epoch history for the 3 training-curve diagrams.
# NB6e doesn't persist `history` to disk by default. If you want the loss /
# accuracy / macro-F1 curves reproduced here too, add this ONE cell to NB6e
# right after its training loop (Step 10), then rerun NB6e once:
#
#     with open(SAVE_DIR / "fullmix_history.json", "w") as f:
#         json.dump(history, f, indent=2)
#
# This notebook will pick it up automatically if present.
history_path = SAVE_DIR / "fullmix_history.json"
HISTORY = None
if history_path.exists():
    with open(history_path) as f:
        HISTORY = json.load(f)
    print("fullmix_history.json found -- training-curve diagrams will be generated.")
else:
    print("fullmix_history.json not found -- the 3 training-curve diagrams will be "
          "skipped (see the note above to enable them). All other diagrams are unaffected.")

## Step 7 — Run inference once on every split

In [ ]:
def evaluate_array(X, y, M, split_name, batch_size=64):
    X_t = torch.from_numpy(X.transpose(0, 2, 1))
    M_t = torch.from_numpy(M)
    y_t = torch.from_numpy(y)
    loader = DataLoader(TensorDataset(X_t, M_t, y_t), batch_size=batch_size,
                         shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)

    all_preds, all_probs, all_true = [], [], []
    with torch.no_grad():
        for xb, mb, yb in loader:
            out = model(xb.to(device), mb.to(device))
            probs = torch.softmax(out["risk"], dim=1).cpu().numpy()
            all_preds.extend(probs.argmax(axis=1))
            all_probs.append(probs)
            all_true.extend(yb.numpy())

    preds = np.array(all_preds)
    true = np.array(all_true)
    probs = np.vstack(all_probs)

    acc = (preds == true).mean()
    mf1 = f1_score(true, preds, average="macro", zero_division=0)
    wf1 = f1_score(true, preds, average="weighted", zero_division=0)
    per_class_f1 = f1_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])
    per_class_prec = precision_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])
    per_class_rec = recall_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])

    print(f"{split_name:28s} | n={len(true):6,}  Acc={acc*100:6.2f}%  MacroF1={mf1:.4f}  WeightedF1={wf1:.4f}")

    return {
        "split": split_name, "accuracy": float(acc), "macro_f1": float(mf1), "weighted_f1": float(wf1),
        "per_class_f1": per_class_f1, "per_class_prec": per_class_prec, "per_class_rec": per_class_rec,
        "n_samples": int(len(true)), "preds": preds, "true": true, "probs": probs,
    }


print("Running inference on all five splits...\n")
results_ptbxl   = evaluate_array(X_ptb_test, y_ptb_test, M_ptb_test, "PTB-XL Test")
results_incart  = evaluate_array(X_incart_test, y_incart_test, M_incart_test, "INCART Test")
results_cpsc    = evaluate_array(X_cpsc_test, y_cpsc_test, M_cpsc_test, "CPSC2018 Test")
results_chapman = evaluate_array(X_chapman_test, y_chapman_test, M_chapman_test, "Chapman Test")
results_mix     = evaluate_array(X_mix_test, y_mix_test, M_mix_test, "Pooled Mix Test")

split_results = [
    (results_ptbxl,   "PTB-XL",     "steelblue"),
    (results_incart,  "INCART",     "seagreen"),
    (results_cpsc,    "CPSC2018",   "firebrick"),
    (results_chapman, "Chapman",    "purple"),
    (results_mix,     "Pooled Mix", "darkorange"),
]
print("\nInference complete. Ready to render diagrams individually below.")

## Diagram 1 — Training Loss curve

In [ ]:
if HISTORY is not None:
    epochs_ran = range(1, len(HISTORY["train_loss"]) + 1)
    plt.figure(figsize=(9, 5.5))
    plt.plot(epochs_ran, HISTORY["train_loss"], label="Train Loss", color="steelblue")
    plt.plot(epochs_ran, HISTORY["val_loss"], label="Val Loss", color="orange")
    plt.title("Full-Mix Training -- Loss", fontweight="bold")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(DIAG_DIR / "fullmix_diag_train_val_loss.png", dpi=600)
    plt.show()
    print("Saved fullmix_diag_train_val_loss.png")
else:
    print("Skipped -- fullmix_history.json not found (see Step 6 note).")

## Diagram 2 — Training Accuracy curve

In [ ]:
if HISTORY is not None:
    epochs_ran = range(1, len(HISTORY["train_acc"]) + 1)
    plt.figure(figsize=(9, 5.5))
    plt.plot(epochs_ran, HISTORY["train_acc"], label="Train Acc", color="steelblue")
    plt.plot(epochs_ran, HISTORY["val_acc"], label="Val Acc", color="orange")
    plt.title("Full-Mix Training -- Accuracy", fontweight="bold")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(DIAG_DIR / "fullmix_diag_train_val_accuracy.png", dpi=600)
    plt.show()
    print("Saved fullmix_diag_train_val_accuracy.png")
else:
    print("Skipped -- fullmix_history.json not found (see Step 6 note).")

## Diagram 3 — Training Macro-F1 curve

In [ ]:
if HISTORY is not None:
    epochs_ran = range(1, len(HISTORY["train_f1"]) + 1)
    best_ep = int(np.argmax(HISTORY["val_f1"])) + 1
    plt.figure(figsize=(9, 5.5))
    plt.plot(epochs_ran, HISTORY["train_f1"], label="Train F1", color="steelblue")
    plt.plot(epochs_ran, HISTORY["val_f1"], label="Val F1", color="orange")
    plt.axvline(best_ep, color="green", linestyle="--", alpha=0.6, label=f"Best (ep {best_ep})")
    plt.title("Full-Mix Training -- Macro F1", fontweight="bold")
    plt.xlabel("Epoch"); plt.ylabel("Macro F1")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(DIAG_DIR / "fullmix_diag_train_val_macrof1.png", dpi=600)
    plt.show()
    print("Saved fullmix_diag_train_val_macrof1.png")
else:
    print("Skipped -- fullmix_history.json not found (see Step 6 note).")

## Diagram 4 — Per-Class F1 Score (all 5 splits)

In [ ]:
x = np.arange(4)
width = 0.16
plt.figure(figsize=(10, 6))
for i, (res, sname, color) in enumerate(split_results):
    vals = res["per_class_f1"]
    bars = plt.bar(x + i * width, vals, width, label=sname, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.2f}", ha="center", fontsize=7)
plt.xticks(x + width * 2, CLASS_NAMES)
plt.title("Per-Class F1 Score -- Full-Mix Model", fontweight="bold")
plt.ylim(0, 1.15)
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_perclass_f1.png", dpi=600)
plt.show()
print("Saved fullmix_diag_perclass_f1.png")

## Diagram 5 — Per-Class Precision (all 5 splits)

In [ ]:
x = np.arange(4)
width = 0.16
plt.figure(figsize=(10, 6))
for i, (res, sname, color) in enumerate(split_results):
    vals = res["per_class_prec"]
    bars = plt.bar(x + i * width, vals, width, label=sname, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.2f}", ha="center", fontsize=7)
plt.xticks(x + width * 2, CLASS_NAMES)
plt.title("Per-Class Precision -- Full-Mix Model", fontweight="bold")
plt.ylim(0, 1.15)
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_perclass_precision.png", dpi=600)
plt.show()
print("Saved fullmix_diag_perclass_precision.png")

## Diagram 6 — Per-Class Recall (all 5 splits)

In [ ]:
x = np.arange(4)
width = 0.16
plt.figure(figsize=(10, 6))
for i, (res, sname, color) in enumerate(split_results):
    vals = res["per_class_rec"]
    bars = plt.bar(x + i * width, vals, width, label=sname, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.2f}", ha="center", fontsize=7)
plt.xticks(x + width * 2, CLASS_NAMES)
plt.title("Per-Class Recall -- Full-Mix Model", fontweight="bold")
plt.ylim(0, 1.15)
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_perclass_recall.png", dpi=600)
plt.show()
print("Saved fullmix_diag_perclass_recall.png")

## Diagram 7 — Confusion Matrix: PTB-XL

In [ ]:
res, sname, _ = split_results[0]
cm = confusion_matrix(res["true"], res["preds"], labels=[0, 1, 2, 3])
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix -- {sname}  (Acc={res['accuracy']*100:.1f}%, n={res['n_samples']:,})", fontweight="bold")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_cm_ptbxl.png", dpi=600)
plt.show()
print("Saved fullmix_diag_cm_ptbxl.png")

## Diagram 8 — Confusion Matrix: INCART

In [ ]:
res, sname, _ = split_results[1]
cm = confusion_matrix(res["true"], res["preds"], labels=[0, 1, 2, 3])
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix -- {sname}  (Acc={res['accuracy']*100:.1f}%, n={res['n_samples']:,})", fontweight="bold")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_cm_incart.png", dpi=600)
plt.show()
print("Saved fullmix_diag_cm_incart.png")

## Diagram 9 — Confusion Matrix: CPSC 2018

In [ ]:
res, sname, _ = split_results[2]
cm = confusion_matrix(res["true"], res["preds"], labels=[0, 1, 2, 3])
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix -- {sname}  (Acc={res['accuracy']*100:.1f}%, n={res['n_samples']:,})", fontweight="bold")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_cm_cpsc2018.png", dpi=600)
plt.show()
print("Saved fullmix_diag_cm_cpsc2018.png")

## Diagram 10 — Confusion Matrix: Chapman-Shaoxing

In [ ]:
res, sname, _ = split_results[3]
cm = confusion_matrix(res["true"], res["preds"], labels=[0, 1, 2, 3])
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix -- {sname}  (Acc={res['accuracy']*100:.1f}%, n={res['n_samples']:,})", fontweight="bold")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_cm_chapman.png", dpi=600)
plt.show()
print("Saved fullmix_diag_cm_chapman.png")

## Diagram 11 — Confusion Matrix: Pooled Mix

In [ ]:
res, sname, _ = split_results[4]
cm = confusion_matrix(res["true"], res["preds"], labels=[0, 1, 2, 3])
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix -- {sname}  (Acc={res['accuracy']*100:.1f}%, n={res['n_samples']:,})", fontweight="bold")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_cm_pooled.png", dpi=600)
plt.show()
print("Saved fullmix_diag_cm_pooled.png")

## Diagram 12 — ROC Curve: PTB-XL

In [ ]:
res, sname, _ = split_results[0]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, k], res["probs"][:, k])
        auc_k = roc_auc_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    except Exception:
        pass
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.title(f"ROC Curves -- {sname}", fontweight="bold")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_roc_ptbxl.png", dpi=600)
plt.show()
print("Saved fullmix_diag_roc_ptbxl.png")

## Diagram 13 — ROC Curve: INCART

In [ ]:
res, sname, _ = split_results[1]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, k], res["probs"][:, k])
        auc_k = roc_auc_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    except Exception:
        pass
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.title(f"ROC Curves -- {sname}", fontweight="bold")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_roc_incart.png", dpi=600)
plt.show()
print("Saved fullmix_diag_roc_incart.png")

## Diagram 14 — ROC Curve: CPSC 2018

In [ ]:
res, sname, _ = split_results[2]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, k], res["probs"][:, k])
        auc_k = roc_auc_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    except Exception:
        pass
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.title(f"ROC Curves -- {sname}", fontweight="bold")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_roc_cpsc2018.png", dpi=600)
plt.show()
print("Saved fullmix_diag_roc_cpsc2018.png")

## Diagram 15 — ROC Curve: Chapman-Shaoxing

In [ ]:
res, sname, _ = split_results[3]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, k], res["probs"][:, k])
        auc_k = roc_auc_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    except Exception:
        pass
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.title(f"ROC Curves -- {sname}", fontweight="bold")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_roc_chapman.png", dpi=600)
plt.show()
print("Saved fullmix_diag_roc_chapman.png")

## Diagram 16 — ROC Curve: Pooled Mix

In [ ]:
res, sname, _ = split_results[4]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, k], res["probs"][:, k])
        auc_k = roc_auc_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    except Exception:
        pass
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.title(f"ROC Curves -- {sname}", fontweight="bold")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_roc_pooled.png", dpi=600)
plt.show()
print("Saved fullmix_diag_roc_pooled.png")

## Diagram 17 — Precision-Recall Curve: PTB-XL

In [ ]:
res, sname, _ = split_results[0]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        prec, rec, _ = precision_recall_curve(y_bin[:, k], res["probs"][:, k])
        ap_k = average_precision_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    except Exception:
        pass
plt.title(f"Precision-Recall Curves -- {sname}", fontweight="bold")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_pr_ptbxl.png", dpi=600)
plt.show()
print("Saved fullmix_diag_pr_ptbxl.png")

## Diagram 18 — Precision-Recall Curve: INCART

In [ ]:
res, sname, _ = split_results[1]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        prec, rec, _ = precision_recall_curve(y_bin[:, k], res["probs"][:, k])
        ap_k = average_precision_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    except Exception:
        pass
plt.title(f"Precision-Recall Curves -- {sname}", fontweight="bold")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_pr_incart.png", dpi=600)
plt.show()
print("Saved fullmix_diag_pr_incart.png")

## Diagram 19 — Precision-Recall Curve: CPSC 2018

In [ ]:
res, sname, _ = split_results[2]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        prec, rec, _ = precision_recall_curve(y_bin[:, k], res["probs"][:, k])
        ap_k = average_precision_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    except Exception:
        pass
plt.title(f"Precision-Recall Curves -- {sname}", fontweight="bold")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_pr_cpsc2018.png", dpi=600)
plt.show()
print("Saved fullmix_diag_pr_cpsc2018.png")

## Diagram 20 — Precision-Recall Curve: Chapman-Shaoxing

In [ ]:
res, sname, _ = split_results[3]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        prec, rec, _ = precision_recall_curve(y_bin[:, k], res["probs"][:, k])
        ap_k = average_precision_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    except Exception:
        pass
plt.title(f"Precision-Recall Curves -- {sname}", fontweight="bold")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_pr_chapman.png", dpi=600)
plt.show()
print("Saved fullmix_diag_pr_chapman.png")

## Diagram 21 — Precision-Recall Curve: Pooled Mix

In [ ]:
res, sname, _ = split_results[4]
y_bin = label_binarize(res["true"], classes=[0, 1, 2, 3])
plt.figure(figsize=(7, 6))
for k, color in zip(range(4), CLASS_COLORS):
    try:
        prec, rec, _ = precision_recall_curve(y_bin[:, k], res["probs"][:, k])
        ap_k = average_precision_score(y_bin[:, k], res["probs"][:, k])
        plt.plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    except Exception:
        pass
plt.title(f"Precision-Recall Curves -- {sname}", fontweight="bold")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_pr_pooled.png", dpi=600)
plt.show()
print("Saved fullmix_diag_pr_pooled.png")

## Diagram 22 — Final Comparison: Accuracy across all splits and prior notebooks

In [ ]:
def _load_json(path):
    p = SAVE_DIR / path
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return {}

tr_nb4   = _load_json("training_results.json")
jr_nb6c  = _load_json("joint_training_results.json")
cross_nb6d = _load_json("crossdataset_results.json")

rows = [
    ("PTB-XL (NB4)", tr_nb4.get("test_accuracy", 0), tr_nb4.get("test_macro_f1", 0)),
    ("PTB-XL+INCART Joint (NB6c)", jr_nb6c.get("pooled_val", {}).get("accuracy", 0),
     jr_nb6c.get("pooled_val", {}).get("macro_f1", 0)),
    ("CPSC2018 zero-shot (NB6d)", cross_nb6d.get("cpsc2018_accuracy", 0), cross_nb6d.get("cpsc2018_macro_f1", 0)),
    ("Chapman zero-shot (NB6d)", cross_nb6d.get("chapman_accuracy", 0), cross_nb6d.get("chapman_macro_f1", 0)),
    ("PTB-XL Test (Full-Mix)", results_ptbxl["accuracy"], results_ptbxl["macro_f1"]),
    ("INCART Test (Full-Mix)", results_incart["accuracy"], results_incart["macro_f1"]),
    ("CPSC2018 Test (Full-Mix)", results_cpsc["accuracy"], results_cpsc["macro_f1"]),
    ("Chapman Test (Full-Mix)", results_chapman["accuracy"], results_chapman["macro_f1"]),
    ("Pooled Mix Test (Full-Mix)", results_mix["accuracy"], results_mix["macro_f1"]),
]
names = [r[0] for r in rows]
accs = [r[1] * 100 for r in rows]
bar_cols = (["gray"] * 4) + (["darkorange"] * 5)

plt.figure(figsize=(11, 6))
bars = plt.bar(names, accs, color=bar_cols)
for bar, val in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val:.1f}%", ha="center", fontsize=8, fontweight="bold")
plt.title("Accuracy -- Across All Training Strategies", fontweight="bold")
plt.ylabel("Accuracy (%)"); plt.ylim(0, 115)
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_comparison_accuracy.png", dpi=600)
plt.show()
print("Saved fullmix_diag_comparison_accuracy.png")

## Diagram 23 — Final Comparison: Macro F1 across all splits and prior notebooks

In [ ]:
f1s = [r[2] for r in rows]

plt.figure(figsize=(11, 6))
bars = plt.bar(names, f1s, color=bar_cols)
for bar, val in zip(bars, f1s):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")
plt.title("Macro F1 -- Across All Training Strategies", fontweight="bold")
plt.ylabel("Macro F1"); plt.ylim(0, 1.15)
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(DIAG_DIR / "fullmix_diag_comparison_macrof1.png", dpi=600)
plt.show()
print("Saved fullmix_diag_comparison_macrof1.png")

print()
print("NOTEBOOK 6f COMPLETE")
print("=" * 55)
print(f"All individual diagrams saved under: {DIAG_DIR.resolve()}")